
# KNN Geo CV Tuning

KNN tuning on the same outer training set as the GP workflow, using explicit inner geographic CV folds.


In [2]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline

CWD = Path.cwd()
if CWD.name == "gridsearch":
    NOTEBOOK_DIR = CWD
elif (CWD / "gridsearch").exists() and CWD.name == "code":
    NOTEBOOK_DIR = CWD / "gridsearch"
elif (CWD / "code" / "gridsearch").exists():
    NOTEBOOK_DIR = CWD / "code" / "gridsearch"
else:
    NOTEBOOK_DIR = CWD
CODE_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(CODE_DIR))

from eval_workflow import make_geo_cv_index_pairs, write_result_row
from knn_features import WifiKNNFeatureTransformer


In [3]:

SPLIT_ROOT = CODE_DIR / "eval_splits" / "geo_outer0"
INNER_GEO5 = SPLIT_ROOT / "inner_geo5"
RESULT_PATH = CODE_DIR / "eval_results" / "knn_final_holdout.csv"
CV_RESULTS_PATH = CODE_DIR / "eval_results" / "knn_geo5_cv_results.csv"

X_outer_train = np.load(SPLIT_ROOT / "outer_train_X.npy")
y_outer_train = np.load(SPLIT_ROOT / "outer_train_y.npy")
outer_train_idx = np.load(SPLIT_ROOT / "outer_train_idx.npy")

X_outer_holdout = np.load(SPLIT_ROOT / "outer_holdout_X.npy")
y_outer_holdout = np.load(SPLIT_ROOT / "outer_holdout_y.npy")

n_ap = int(np.nanmax(X_outer_train[:, 4])) + 1


In [4]:

index_to_position = {int(idx): pos for pos, idx in enumerate(outer_train_idx)}
geo_cv_global = make_geo_cv_index_pairs(INNER_GEO5, 5)
geo_cv = [
    (
        np.array([index_to_position[int(idx)] for idx in train_idx], dtype=int),
        np.array([index_to_position[int(idx)] for idx in test_idx], dtype=int),
    )
    for train_idx, test_idx in geo_cv_global
]
[(len(train_idx), len(test_idx)) for train_idx, test_idx in geo_cv]


[(1589, 475), (1666, 398), (1620, 444), (1690, 374), (1691, 373)]

In [11]:

param_grid = {
    "features__indoor_scale": [0.5, 1.0, 2.0],
    "features__ap_scale": [0.25, 0.5, 1.0],
    "knn__n_neighbors": np.arange(3, 60, 3),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

knn_pipeline = Pipeline([
    ("features", WifiKNNFeatureTransformer(n_ap=n_ap)),
    ("knn", KNeighborsRegressor()),
])

knn_grid = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=geo_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)


In [12]:

knn_grid.fit(X_outer_train, y_outer_train)
cv_results = pd.DataFrame(knn_grid.cv_results_)
CV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(CV_RESULTS_PATH, index=False)
knn_grid.best_params_, -knn_grid.best_score_


Fitting 5 folds for each of 684 candidates, totalling 3420 fits


({'features__ap_scale': 1.0,
  'features__indoor_scale': 0.5,
  'knn__n_neighbors': np.int64(54),
  'knn__p': 1,
  'knn__weights': 'distance'},
 np.float64(79.94013707806342))

In [13]:

preds = knn_grid.predict(X_outer_holdout)
final_mse = float(np.mean((preds - y_outer_holdout) ** 2))
row = {
    "model": "knn",
    "mse": final_mse,
    "inner_geo5_mse": float(-knn_grid.best_score_),
    "n_train": int(X_outer_train.shape[0]),
    "n_test": int(X_outer_holdout.shape[0]),
    **knn_grid.best_params_,
}
write_result_row(RESULT_PATH, row)
row


{'model': 'knn',
 'mse': 84.25625594486999,
 'inner_geo5_mse': 79.94013707806342,
 'n_train': 2064,
 'n_test': 553,
 'features__ap_scale': 1.0,
 'features__indoor_scale': 0.5,
 'knn__n_neighbors': np.int64(54),
 'knn__p': 1,
 'knn__weights': 'distance'}